In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import torch.nn.functional as F
import torch.optim as optim
from torchvision import transforms

D:\ANACONDA\Lib\site-packages\torch\cuda\__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [3]:
import numpy as np
import pandas as pd

In [7]:
!pip install kagglehub

In [9]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("puneet6060/intel-image-classification")

print("Path to dataset files:", path)

100%|██████████| 346M/346M [02:54<00:00, 2.08MB/s] 

Extracting files...


Path to dataset files: C:\Users\Prabhsimrat Singh\.cache\kagglehub\datasets\puneet6060\intel-image-classification\versions\2


In [15]:
from torchvision import datasets


In [53]:
train_transform=transforms.Compose([
    transforms.Resize((128,128)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.5,0.5,0.5],
        [0.5,0.5,0.5]
    )
])

test_transform=transforms.Compose([
    transforms.Resize((128,128)),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.5,0.5,0.5],
        [0.5,0.5,0.5]
    )
])

In [55]:
print(path+"\seg_train"+"\seg_train")

C:\Users\Prabhsimrat Singh\.cache\kagglehub\datasets\puneet6060\intel-image-classification\versions\2\seg_train\seg_train


<>:1: SyntaxWarning: invalid escape sequence '\s'
<>:1: SyntaxWarning: invalid escape sequence '\s'
<>:1: SyntaxWarning: invalid escape sequence '\s'
<>:1: SyntaxWarning: invalid escape sequence '\s'
C:\Users\Prabhsimrat Singh\AppData\Local\Temp\ipykernel_4504\4168169384.py:1: SyntaxWarning: invalid escape sequence '\s'
  print(path+"\seg_train"+"\seg_train")
C:\Users\Prabhsimrat Singh\AppData\Local\Temp\ipykernel_4504\4168169384.py:1: SyntaxWarning: invalid escape sequence '\s'
  print(path+"\seg_train"+"\seg_train")


In [57]:
train_dataset=datasets.ImageFolder(
    root=path+"\seg_train"+"\seg_train",
    transform=train_transform
)
print(dataset.classes)

<>:2: SyntaxWarning: invalid escape sequence '\s'
<>:2: SyntaxWarning: invalid escape sequence '\s'
<>:2: SyntaxWarning: invalid escape sequence '\s'
<>:2: SyntaxWarning: invalid escape sequence '\s'
C:\Users\Prabhsimrat Singh\AppData\Local\Temp\ipykernel_4504\440522919.py:2: SyntaxWarning: invalid escape sequence '\s'
  root=path+"\seg_train"+"\seg_train",
C:\Users\Prabhsimrat Singh\AppData\Local\Temp\ipykernel_4504\440522919.py:2: SyntaxWarning: invalid escape sequence '\s'
  root=path+"\seg_train"+"\seg_train",


['buildings', 'forest', 'glacier', 'mountain', 'sea', 'street']


In [59]:
test_dataset=datasets.ImageFolder(
    root=path+"\seg_test"+"\seg_test",
    transform=test_transform
)

<>:2: SyntaxWarning: invalid escape sequence '\s'
<>:2: SyntaxWarning: invalid escape sequence '\s'
<>:2: SyntaxWarning: invalid escape sequence '\s'
<>:2: SyntaxWarning: invalid escape sequence '\s'
C:\Users\Prabhsimrat Singh\AppData\Local\Temp\ipykernel_4504\1045460205.py:2: SyntaxWarning: invalid escape sequence '\s'
  root=path+"\seg_test"+"\seg_test",
C:\Users\Prabhsimrat Singh\AppData\Local\Temp\ipykernel_4504\1045460205.py:2: SyntaxWarning: invalid escape sequence '\s'
  root=path+"\seg_test"+"\seg_test",


In [61]:
print(test_dataset.classes)

['buildings', 'forest', 'glacier', 'mountain', 'sea', 'street']


In [63]:
train_loader=DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

test_loader=DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

In [77]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1=nn.Conv2d(
            in_channels=3,
            out_channels=16,
            kernel_size=3,
            padding=1
        )

        self.conv2=nn.Conv2d(
            in_channels=16,
            out_channels=32,
            kernel_size=3,
            padding=1
        )

        self.conv3=nn.Conv2d(
            in_channels=32,
            out_channels=64,
            kernel_size=3,
            padding=1
        )

        self.pool=nn.MaxPool2d(
            kernel_size=2,
            stride=2
        )

        self.dropout=nn.Dropout(p=0.5)
        
        self.bn1=nn.BatchNorm2d(16)
        
        self.bn2=nn.BatchNorm2d(32)
        
        self.bn3=nn.BatchNorm2d(64)
        
        self.fc1=nn.Linear(
         64*16*16,
            128
        )
        
        self.fc2=nn.Linear(
            128,
            6
        )

    def forward(self,x):
        x=self.pool(F.relu(self.bn1(self.conv1(x))))
        x=self.pool(F.relu(self.bn2(self.conv2(x))))
        x=self.pool(F.relu(self.bn3(self.conv3(x))))

        x=torch.flatten(x,start_dim=1)
        x=F.relu(self.fc1(x))
        x=self.dropout(x)
        x=self.fc2(x)
        return x

In [79]:
model=CNN()

In [81]:
criterion=nn.CrossEntropyLoss()

In [83]:
optimiser=optim.Adam(model.parameters(),lr=0.001)

In [85]:
device=torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

model.to(device)

CNN(
  (conv1): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (dropout): Dropout(p=0.5, inplace=False)
  (bn1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (bn2): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (bn3): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (fc1): Linear(in_features=16384, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=6, bias=True)
)

In [87]:
epochs=5
for epoch in range(epochs):
    model.train()
    epoch_loss=0
    for x_train,y_train in train_loader:
        x_train=x_train.to(device)
        y_train=y_train.to(device)
        optimiser.zero_grad()
        output=model(x_train)
        loss=criterion(output,y_train)
        loss.backward()
        optimiser.step()
        epoch_loss+=loss.item()

    print("loss in epoch ",epoch,": ",epoch_loss)

loss in epoch  0 :  596.6852084994316
loss in epoch  1 :  498.1995857357979
loss in epoch  2 :  468.8142383694649
loss in epoch  3 :  448.89965456724167
loss in epoch  4 :  430.869236767292


In [101]:
model.eval()

y_true=[]
y_pred=[]
with torch.no_grad():

    for x_test,y_test in test_loader:
        x_test=x_test.to(device)
        y_test=y_test.to(device)
        outputs=model(x_test)

        _,predicted=torch.max(outputs,1)

        y_true.extend(y_test.cpu().numpy())
        y_pred.extend(predicted.cpu().numpy())

In [103]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_true,
        y_pred,
        target_names=test_dataset.classes
    )
)

              precision    recall  f1-score   support

   buildings       0.68      0.84      0.75       437
      forest       0.94      0.93      0.94       474
     glacier       0.76      0.66      0.70       553
    mountain       0.60      0.79      0.69       525
         sea       0.61      0.47      0.53       510
      street       0.84      0.73      0.78       501

    accuracy                           0.73      3000
   macro avg       0.74      0.74      0.73      3000
weighted avg       0.74      0.73      0.73      3000



Accuracy: 73.10%
